# BGE-M3 LoRA 파인튜닝 (질의 측만) — Colab T4`Financial-HGT`의 QueryEncoder(잔차 MLP)에 대한 **대조군**을 만든다.지금까지 두 레포 어디에도 BGE 인코더 가중치를 실제로 학습한 arm이 없었고,"임베딩 fine-tuning은 실패"로 알려진 결과는 사실 HGT로 문서 임베딩을 재구축한다른 실험이다 (`analysis/NOTION_UPDATE.md:151`).**공정 비교를 위해 고정하는 것**- 문서(조항) 임베딩은 건드리지 않는다 — MLP arm과 같은 `emb_cache/clause_embs_*.safetensors`- 데이터·분할(seed 42, test_size 300)·손실·hard negative 일정·체크포인트 규칙 동일- 두 방식 모두 **학습 시작 시점이 순수 BGE 베이스라인과 동일**  (MLP: 마지막 Linear 0 초기화 / LoRA: B=0 → BA=0 → W_eff=W_0)**런타임**: 수정 → 런타임 유형 변경 → **T4 GPU**

## 0. 환경 확인

In [1]:
!nvidia-smi
import torch, sys
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "런타임 유형을 T4 GPU로 바꾸세요"
print("GPU:", torch.cuda.get_device_name(0))
# T4는 Turing이라 bf16 미지원 -> fp16 + GradScaler를 쓴다
print("bf16 지원:", torch.cuda.is_bf16_supported())

Fri Jul 31 22:40:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. 의존성 설치

In [2]:
!pip -q install "peft>=0.11.0" "transformers>=4.45.0" "sentence-transformers>=2.7.0" \
                "safetensors>=0.4.0" pandas openpyxl scipy

## 2. 레포 clone`data/`가 git에 추적되고 있어(`nodes.csv`, `triplets.csv`, `for_review_corrected.xlsx`)**데이터 업로드가 필요 없다.** 레포 전체 pack이 15.4MiB뿐이다.

In [3]:
# 비공개 레포면 GitHub Personal Access Token(repo 권한)을 붙여 clone한다.
# 공개 레포면 getpass에서 그냥 엔터.
import os, getpass
BRANCH = "claude/two-repos-project-report-qjhjzu"
OWNER, REPO = "WB-Jang", "Financial-HGT"

if not os.path.exists("/content/Financial-HGT"):
    tok = getpass.getpass("GitHub token (공개 레포면 엔터): ").strip()
    auth = f"{tok}@" if tok else ""
    url = f"https://{auth}github.com/{OWNER}/{REPO}.git"
    !git clone -q -b {BRANCH} {url} /content/Financial-HGT

%cd /content/Financial-HGT
!git log --oneline -1
!ls -la data/

GitHub token (공개 레포면 엔터): ··········
/content/Financial-HGT
ad863ee (HEAD -> claude/two-repos-project-report-qjhjzu, origin/claude/two-repos-project-report-qjhjzu) Extend paired answer analysis with the hybrid breadth-alpha dose-response curve
total 21600
drwxr-xr-x 2 root root    4096 Jul 31 22:41 .
drwxr-xr-x 7 root root    4096 Jul 31 22:41 ..
-rw-r--r-- 1 root root 6029450 Jul 31 22:41 for_review_corrected.xlsx
-rw-r--r-- 1 root root 8647506 Jul 31 22:41 nodes.csv
-rw-r--r-- 1 root root 7425246 Jul 31 22:41 triplets.csv


## 3. ⭐ MLP 체크포인트 업로드 (권장)`query_encoder_best.safetensors` (~4.2MB)는 `.gitignore` 대상이라 레포에 없다.**기존 `pl_hybrid` 답변 결과(REF_avg 0.1710)를 만든 바로 그 모델**이므로,LoRA와의 비교 기준선으로 이 파일을 그대로 써야 한다.재학습으로 대체하면 다른 체크포인트가 나와 기존 답변 결과와의 연결이 끊긴다.없으면 아래 셀을 건너뛰고 4-b에서 재학습할 수 있으나, 그 경우 MLP arm은"재현본"이지 "원본"이 아님을 결과에 명시할 것.

In [4]:
from google.colab import files
import os
if not os.path.exists("query_encoder_best.safetensors"):
    print("query_encoder_best.safetensors 를 선택하세요 (없으면 취소 후 4-b로)")
    up = files.upload()
print("존재:", os.path.exists("query_encoder_best.safetensors"))

query_encoder_best.safetensors 를 선택하세요 (없으면 취소 후 4-b로)


Saving query_encoder_fhgt_best.pt to query_encoder_fhgt_best.pt
존재: False


## 4-a. 조항 임베딩 생성 (캐시)BGE-M3 다운로드(~2.3GB) + 조항 9,311건 인코딩. T4로 3~5분.캐시 키가 텍스트 내용+순서의 md5라 CSV가 같으면 로컬 캐시와 호환된다.

In [6]:
!pip install torch_geometric
import pandas as pd
import torch
from data_loader import normalize_johang_key, fsc_dataset_preprocessing, encode_texts_cached, make_bge_encoder
from retrieval_common import build_clause_index

nodes_df = pd.read_csv('data/nodes.csv')
nodes_df['new_johang'] = [
    normalize_johang_key(l, a, h)
    for l, a, h in zip(nodes_df['law_nm'], nodes_df['article_number'], nodes_df['hang_number'])
]
clause_list, clause_texts = build_clause_index(nodes_df)
print(f"조항 노드 {len(clause_list):,}개")

enc = make_bge_encoder()
clause_embs = encode_texts_cached(enc, clause_texts, 'clause_embs')
fsc = fsc_dataset_preprocessing(file='data/for_review_corrected.xlsx', nodes_df=nodes_df, test_size=300)
from retrieval_common import build_retrieval_items
train_items, _ = build_retrieval_items(fsc[fsc.split=='train'].reset_index(drop=True), clause_list)
test_items,  _ = build_retrieval_items(fsc[fsc.split=='test'].reset_index(drop=True),  clause_list)
_ = encode_texts_cached(enc, [it['query'] for it in train_items], 'fsc_query_embs')
_ = encode_texts_cached(enc, [it['query'] for it in test_items],  'fsc_query_embs')
del enc; torch.cuda.empty_cache()
print(f"train {len(train_items)}건 / test {len(test_items)}건")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.1 MB/s eta 0:00:00
조항 노드 9,311개
  BGE-M3 인코더 로드 (device=cuda)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

  💾 임베딩 캐시 저장: clause_embs_542bc3dadf9b0819.safetensors (36.4 MB)
평가가능 질의: 3,054건 / 전체 고유 질의 3,836건 (test는 이 평가가능 질의에서만 300건 층화추출)
  fsc_query_embs: 2,765건 인코딩 중... (batch=12, 완료 후 캐시에 저장)


Batches:   0%|          | 0/231 [00:00<?, ?it/s]

  💾 임베딩 캐시 저장: fsc_query_embs_ae25839016ef10b1.safetensors (10.8 MB)
  fsc_query_embs: 301건 인코딩 중... (batch=12, 완료 후 캐시에 저장)


Batches:   0%|          | 0/26 [00:00<?, ?it/s]

  💾 임베딩 캐시 저장: fsc_query_embs_32cd9c5594398984.safetensors (1.2 MB)
train 2765건 / test 301건


### 질의 토큰 길이 실측 → `max_len` 확정절단 길이가 SentenceTransformer의 인코딩과 어긋나면 질의와 조항이 다른 공간에놓인다. 아래 분위수를 보고 `MAX_LEN`을 정한다 (게이트 ①이 이를 검증한다).

In [7]:
import numpy as np
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('BAAI/bge-m3')
lens = [len(tok(it['query'])['input_ids']) for it in train_items[:1500]]
print("질의 토큰 길이 분위수:",
      {q: int(np.percentile(lens, q)) for q in (50, 75, 90, 95, 99, 100)})
MAX_LEN = 256
p99 = np.percentile(lens, 99)
if p99 > MAX_LEN:
    MAX_LEN = int(min(512, 32 * np.ceil(p99 / 32)))
    print(f"99분위({p99:.0f})가 256을 넘어 MAX_LEN을 {MAX_LEN}으로 상향")
print("MAX_LEN =", MAX_LEN)

질의 토큰 길이 분위수: {50: 88, 75: 134, 90: 203, 95: 259, 99: 438, 100: 974}
99분위(438)가 256을 넘어 MAX_LEN을 448으로 상향
MAX_LEN = 448


## 4-b. (선택) MLP 재학습3번에서 체크포인트를 올렸다면 **건너뛴다.** 캐시된 임베딩만 곱하므로 수 분이면 끝난다.README §1의 조 단위 MRR .500 → .599 재현 여부를 확인하는 검증 목적으로도 유용하다.

In [ ]:
# !python train_query_encoder.py --test_size 300

## 5. LoRA 학습두 게이트를 먼저 통과해야 학습이 시작된다.- **게이트 ①** 어댑터 항등(B=0) 상태에서 우리 CLS 경로 ≡ SentenceTransformer 임베딩 (코사인 ≥ 0.999)- **게이트 ②** 학습 전 val Hit@15 == 순수 BGE 베이스라인micro_batch 8 × accum 4 = 유효 배치 32. 이 손실은 질의마다 독립항이고 분모가전체 코퍼스라 in-batch negative가 없으므로, 누적 gradient가 배치 32와 수학적으로 동일하다.

In [9]:
!pip install --upgrade torchao
!python lora/train_lora.py \
    --test_size 300 \
    --epochs 12 \
    --micro_batch 8 --accum 4 \
    --lr 1e-4 \
    --lora_r 16 --lora_alpha 32 \
    --max_len {MAX_LEN} \
    --amp 1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
사용 기기: cuda | fp16 autocast: True
평가가능 질의: 3,054건 / 전체 고유 질의 3,836건 (test는 이 평가가능 질의에서만 300건 층화추출)
조항 노드 9,311개 | train 2765건(제외 791) | test 301건(제외 0)
  BGE-M3 인코더 로드 (device=cuda)
Loading weights: 100% 391/391 [00:00<00:00, 30974.44it/s]
  ♻️ 임베딩 캐시 재사용: clause_embs (9,311건) <- clause_embs_542bc3dadf9b0819.safetensors
  ♻️ 임베딩 캐시 재사용: fsc_query_embs 

## 6. 검색 평가 — 3 arm`--query_emb_file`로 LoRA 질의 임베딩을 넣으면, 이후 재랭킹·항/조 접기·출력까지**기존 경로를 그대로 통과**한다. MLP arm과 완전히 같은 처리를 받는다는 뜻이다.`--hybrid`는 `pl_hybrid`(기존 최고 구성)와 맞추기 위한 것이다.

In [ ]:
# baseline (순수 BGE)
!python evaluate_rerank.py --no_query_encoder --hybrid --test_size 300
# MLP
!python evaluate_rerank.py --hybrid --test_size 300
# LoRA
!python evaluate_rerank.py --query_emb_file emb_cache/fsc_query_embs_lora.safetensors \
                           --hybrid --test_size 300
!ls -la eval_results/ | grep paragraph

## 7. 대응표본 검정 (주 지표 = 항 단위)301문항이 세 arm 모두 동일하므로 문항 단위로 짝지어 검정한다.부트스트랩 20,000회 95% CI + Wilcoxon, `num_laws` 기준 전체 / 1-2법 / 3-4법 층화.

In [ ]:
!python lora/paired_retrieval_analysis.py \
    --mlp      "eval_results/rerank_origEmb_stage2_hybrid_none_paragraph_*.csv" \
    --lora     "eval_results/rerank_origEmb_lora_hybrid_none_paragraph_*.csv" \
    --baseline "eval_results/rerank_origEmb_bgeq_hybrid_none_paragraph_*.csv" \
    --level paragraph

조 단위는 **서브 지표**로만 확인한다 (두 세밀도에서 순위가 뒤집히는 경우가 있음).

In [ ]:
!python lora/paired_retrieval_analysis.py \
    --mlp      "eval_results/rerank_origEmb_stage2_hybrid_none_article_*.csv" \
    --lora     "eval_results/rerank_origEmb_lora_hybrid_none_article_*.csv" \
    --baseline "eval_results/rerank_origEmb_bgeq_hybrid_none_article_*.csv" \
    --level article

## 8. 산출물 다운로드| 파일 | 용도 ||---|---|| `lora/checkpoints/` | LoRA 어댑터 (~6MB) || `emb_cache/fsc_query_embs_lora.safetensors` | LoRA 질의 임베딩 — 이후 검색 평가를 GPU 없이 로컬에서 돌릴 수 있다 || `lora/results/` · `eval_results/` | 지표 CSV/JSON |

In [ ]:
!mkdir -p /content/out && cp -r lora/checkpoints lora/results /content/out/ \
  && cp emb_cache/fsc_query_embs_lora.safetensors /content/out/ \
  && cp eval_results/*paragraph*.csv eval_results/*article*.csv /content/out/ 2>/dev/null
!cd /content && zip -qr lora_artifacts.zip out && ls -lh lora_artifacts.zip
from google.colab import files
files.download('/content/lora_artifacts.zip')